# Qutrit QPT Bar Chart

This notebook generates the four-panel bar chart figure from the manuscript comparing
raw, error-mitigated, and ideal qutrit process tomography outcome probabilities for a
single metric operator, along with their difference.

Input data is read from `data/raw_tomography/qutrit_raw_counts/`. The figure is saved
to `figures/`.

Set `k` (0-indexed) to select which of the 40 metric operators to plot.

In [ ]:
import os
from math import pi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Parameters

In [ ]:
# Index of the metric operator to plot (0-indexed, 0-39)
k = 39

INPUT_DIR = "data/raw_tomography/qutrit_raw_counts"
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Colour cycle for qutrit states |0>, |1>, |2>
colorcycle = ["#00B5AD", "#EF476F", "#FFC504"]
total_count = 1e4

## Metric operator parameter grid and target unitary

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid


def RY12(angle):
    """Rotation matrix on the 1-2 subspace of the qutrit."""
    c, s = np.cos(angle/2), np.sin(angle/2)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def RY01(angle):
    """Rotation matrix on the 0-1 subspace of the qutrit."""
    c, s = np.cos(angle/2), np.sin(angle/2)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def RZ01(angle):
    """Phase rotation on the 0-1 subspace of the qutrit."""
    return np.array([
        [np.exp(-1j * angle / 2), 0, 0],
        [0, np.exp(1j * angle / 2), 0],
        [0, 0, 1]
    ])


def Utarget(vector):
    """Construct the 3x3 target qutrit unitary for a given metric operator parameter vector.

    The unitary is decomposed into RZ01 and RY01/RY12 rotations parameterised by
    theta, phi, and alpha from the metric operator.

    Args:
        vector: [r, theta, phi, alpha] parameter vector.

    Returns:
        U: (3, 3) complex array, the target qutrit unitary.
    """
    theta, phi, alpha = vector[1], vector[2], vector[3]
    U = (RZ01(-phi)
         .dot(RY01(-theta))
         .dot(RY12(-(np.pi - alpha)))
         .dot(RY01(theta))
         .dot(RZ01(phi)))
    return U

## Compute ideal output probabilities

In [ ]:
grid = eta_grid()
target_u = Utarget(grid[k])

# Six input states: X+, X-, Y+, Y-, Z+, Z- (in qutrit space)
init_states = [
    [1/np.sqrt(2),  1/np.sqrt(2),   0],   # X+
    [1/np.sqrt(2), -1/np.sqrt(2),   0],   # X-
    [1/np.sqrt(2),  1j/np.sqrt(2),  0],   # Y+
    [1/np.sqrt(2), -1j/np.sqrt(2),  0],   # Y-
    [1, 0, 0],                             # Z+
    [0, 1, 0],                             # Z-
]

# Measurement bases: X, Y, Z
meas_x = np.array([[1/np.sqrt(2),  1/np.sqrt(2), 0],
                    [1/np.sqrt(2), -1/np.sqrt(2), 0],
                    [0, 0, 1]])
meas_y = np.array([[1/np.sqrt(2), -1j/np.sqrt(2), 0],
                    [1/np.sqrt(2),  1j/np.sqrt(2), 0],
                    [0, 0, 1]])
meas_z = np.eye(3)
proj_axes = [meas_x, meas_y, meas_z]

# Compute ideal output probabilities for each (input state, measurement basis) pair
expected_outputs = []
for state in init_states:
    for basis in proj_axes:
        final_state = basis.dot(target_u).dot(state)
        final_prob = np.real(final_state * np.conj(final_state))
        expected_outputs.append(final_prob.tolist())

## Load and process experimental data

In [ ]:
data_path = os.path.join(INPUT_DIR, f"qutrit_tomography_metric_{k+1:02d}.csv")
data = pd.read_csv(data_path).values.tolist()

# Clip residual negative REM probabilities and renormalise
for idx in range(18):
    rem_counts = [max(0, p) for p in data[idx][3:6]]
    total = sum(rem_counts)
    data[idx][3:6] = [p / total for p in rem_counts]

results           = [data[i][0:3] for i in range(18)]   # raw counts
results_corrected = [data[i][3:6] for i in range(18)]   # REM-corrected counts
diff_output       = np.array(expected_outputs) - np.array(results_corrected)

## Plot

In [ ]:
import matplotlib.ticker as ticker

# Y-axis labels: one per (input state, measurement basis) combination
axes_labels = ['X', 'Y', 'Z']
sign_labels = ['+', '\N{MINUS SIGN}']
labels = [
    fr"${a0}_{s} \rightarrow m_{{{a1}}}$"
    for a0 in axes_labels
    for s in sign_labels
    for a1 in axes_labels
]

fig, axs = plt.subplots(1, 4, figsize=(12, 6))

datasets = [results, results_corrected, expected_outputs]
titles   = ["Raw Data", "Error-mitigated Data", "Ideal Data"]

# Panels (a)-(c): stacked horizontal bar charts
for i, (ax, dataset, title) in enumerate(zip(axs[:3], datasets, titles)):
    p0 = np.array(dataset)[:, 0]
    p1 = np.array(dataset)[:, 1]
    p2 = np.array(dataset)[:, 2]

    ax.barh(range(len(p0)), p0 * total_count, color=colorcycle[0])
    ax.barh(range(len(p1)), p1 * total_count, left=p0 * total_count, color=colorcycle[1])
    ax.barh(range(len(p2)), p2 * total_count,
            left=(p0 + p1) * total_count, color=colorcycle[2])
    ax.set_xlim([0, total_count])
    ax.set_title(f"({chr(97+i)})  {title}", loc='left', fontweight='bold')
    ax.set_xlabel("Counts")
    ax.grid(axis='x', linestyle='--', alpha=0.5, color='gray')

# Y-axis labels on first panel only
axs[0].set_yticks(range(len(labels)))
axs[0].set_yticklabels(labels)
axs[1].get_yaxis().set_visible(False)
axs[2].get_yaxis().set_visible(False)

# Panel (d): difference plot
p0 = np.array(results_corrected)[:, 0]
p1 = np.array(results_corrected)[:, 1]
p2 = np.array(results_corrected)[:, 2]
axs[3].plot(diff_output[:, 0] * total_count, range(len(p0)),
            marker='o', linestyle='None', color=colorcycle[0])
axs[3].plot(diff_output[:, 1] * total_count, range(len(p1)),
            marker='o', linestyle='None', color=colorcycle[1])
axs[3].plot(diff_output[:, 2] * total_count, range(len(p2)),
            marker='o', linestyle='None', color=colorcycle[2])
axs[3].set_title("(d)  Ideal $-$ Mitigated", loc='left', fontweight='bold')
axs[3].set_xlabel("Difference (Counts)")
axs[3].set_xlim([-0.08 * total_count, 0.08 * total_count])
axs[3].set_yticks(range(len(labels)))
axs[3].set_yticklabels([])
axs[3].tick_params(axis='y', direction='in', length=0)
axs[3].grid(True, axis='both')

# Shared legend
legend_elements = [
    mpatches.Patch(facecolor=colorcycle[0], label=r"$|0\rangle$"),
    mpatches.Patch(facecolor=colorcycle[1], label=r"$|1\rangle$"),
    mpatches.Patch(facecolor=colorcycle[2], label=r"$|2\rangle$"),
]
fig.legend(handles=legend_elements, loc='upper center',
           bbox_to_anchor=(0.5, 1.05), ncol=3, frameon=False, fontsize=12)

plt.tight_layout()

out_path = os.path.join(OUTPUT_DIR, f"qutrit_bar_chart_metric_{k+1:02d}.svg")
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Figure saved to {out_path}")